In [23]:
# MVskew results

# d = 6 analysis

In [24]:
import numpy as np
import pickle 

# SHD
from cdt.metrics import SHD

# SID
import os
os.environ["R_ENABLE_JIT"] = "0"
os.environ["R_HOME"] = r"C:\Program Files\R\R-4.4.2"
from rpy2.robjects.packages import importr
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri, default_converter

def calculate_SID_with_R(W_true, W_est):
    """Calculate structural intervention distance (SID) between two DAGs.

    Args:
        W_true (np.ndarray): [d, d] true adj matrix of DAG
        W_est (np.ndarray): [d, d] estimated adj matrix of DAG
    Returns:
        SID (float): structural intervention distance
    """
    # convert numpy to R matrix
    np_cv_rules = default_converter + numpy2ri.converter
    with np_cv_rules.context():
        nr, nc = W_true.shape
        W_true = robjects.r.matrix(W_true, nrow=nr, ncol=nc)
        robjects.r.assign("W_true_", W_true)

        nr, nc = W_est.shape
        W_est = robjects.r.matrix(W_est, nrow=nr, ncol=nc)
        robjects.r.assign("W_est_", W_est)

    # calculate SID with R package SID
    SID = importr("SID")
    return SID.structIntervDist(robjects.r["W_true_"], robjects.r["W_est_"])[0][0]

In [25]:
from pathlib import Path

N = 100

result_dict = {}
GT_dict = {}

for r in range(N):
    pid = r+1
    with open(f"data/MVskew6/MVskew{pid}.pkl", "rb") as file:
        XB = pickle.load(file)
    X, B = XB["X"], XB["B"]
    GT_dict[pid] = B
    save_file = (
            Path("experiments")
            / "results"
            / "MVskew6"
            / f"result_{pid}.pkl"
        )
    with open(save_file, "rb") as file:
            result = pickle.load(file)
    result_dict[pid] = result

In [26]:
# with PC:
shd_KDE = []
sid_KDE = []
shd_N = []
sid_N = []
for r in range(N):
    pid = r+1
    GT = GT_dict[pid]
    res = result_dict[pid]
    shd_N.append(SHD(target=GT, pred=res["best_graph_N"]))
    sid_N.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_N"]))
    shd_KDE.append(SHD(target=GT, pred=res["best_graph_KDE"]))
    sid_KDE.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_KDE"]))

print("SHD KDE: ", np.mean(shd_KDE))
print("SID KDE: ", np.mean(sid_KDE))
print("SHD N: ", np.mean(shd_N))
print("SID N: ", np.mean(sid_N))

SHD KDE:  2.87
SID KDE:  5.89
SHD N:  2.27
SID N:  4.35


In [27]:
import csv

shd_KDE6 = shd_KDE
sid_KDE6 = sid_KDE
shd_N6 = shd_N
sid_N6 = sid_N

In [28]:
# with orcale MEC:
N = 100

result_dict_oracle = {}

for r in range(N):
    pid = 100+r+1
    save_file = (
            Path("experiments")
            / "results"
            / "MVskew6"
            / f"result_{pid}.pkl"
        )
    with open(save_file, "rb") as file:
            result = pickle.load(file)
    result_dict_oracle[pid] = result


shd_KDE = []
sid_KDE = []
shd_N = []
sid_N = []
for r in range(N):
    pid = r+1
    GT = GT_dict[pid]
    res = result_dict_oracle[pid+100]
    shd_N.append(SHD(target=GT, pred=res["best_graph_N"]))
    sid_N.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_N"]))
    shd_KDE.append(SHD(target=GT, pred=res["best_graph_KDE"]))
    sid_KDE.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_KDE"]))

print("SHD KDE: ", np.mean(shd_KDE))
print("SID KDE: ", np.mean(sid_KDE))
print("SHD N: ", np.mean(shd_N))
print("SID N: ", np.mean(sid_N))

SHD KDE:  0.88
SID KDE:  2.07
SHD N:  0.14
SID N:  0.23


In [29]:
shd_KDE6_O = shd_KDE
sid_KDE6_O = sid_KDE
shd_N6_O = shd_N
sid_N6_O = sid_N

In [30]:
# Check accuracy of nodes that were directed by skewd_MV when using the oracle MEC:
from src.graphutils import undirected_edges, correct_edges


In [31]:
undir_edges = {}
cor_edges = {}
accs = []
total_no_edges = 0
correct_no_edges = 0

for r in range(N):
    pid = r+1
    GT_graph = GT_dict[pid]
    oracle_MEC = result_dict_oracle[pid+100]["MEC"]
    predicted_oracle_based = result_dict_oracle[pid+100]["best_graph_N"]
    undir_edges[r] = undirected_edges(oracle_MEC)
    cor_edges[r] = correct_edges(edges=undir_edges[r], GT_graph=GT_graph, pred_graph=predicted_oracle_based)
    if len(undir_edges[r]) != 0:
        accs.append(len(cor_edges[r])/len(undir_edges[r]))
        if(len(cor_edges[r])/len(undir_edges[r])) != 1:
            print(r)
    total_no_edges += len(undir_edges[r])
    correct_no_edges += len(cor_edges[r])


5
31
45
50
64
71
96


In [32]:
np.mean(accs)

0.9843137254901961

In [33]:
correct_no_edges/total_no_edges

0.9727626459143969

In [34]:
total_no_edges/100 # 2.57 edges per setting that are directed by skewd w/o PC

2.57

# d=10 analysis

In [35]:
from pathlib import Path

N = 100

result_dict = {}
GT_dict = {}

for r in range(N):
    pid = r+1
    with open(f"data/MVskew10/MVskew{pid}.pkl", "rb") as file:
        XB = pickle.load(file)
    X, B = XB["X"], XB["B"]
    GT_dict[pid] = B
    save_file = (
            Path("experiments")
            / "results"
            / "MVskew10"
            / f"result_{pid}.pkl"
        )
    with open(save_file, "rb") as file:
            result = pickle.load(file)
    result_dict[pid] = result

In [36]:
# with PC:
shd_KDE = []
sid_KDE = []
shd_N = []
sid_N = []
for r in range(N):
    pid = r+1
    GT = GT_dict[pid]
    res = result_dict[pid]
    shd_N.append(SHD(target=GT, pred=res["best_graph_N"]))
    sid_N.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_N"]))
    shd_KDE.append(SHD(target=GT, pred=res["best_graph_KDE"]))
    sid_KDE.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_KDE"]))

print("SHD KDE: ", np.mean(shd_KDE))
print("SID KDE: ", np.mean(sid_KDE))
print("SHD N: ", np.mean(shd_N))
print("SID N: ", np.mean(sid_N))

SHD KDE:  5.58
SID KDE:  17.38
SHD N:  4.72
SID N:  14.97


In [37]:
shd_KDE10 = shd_KDE
sid_KDE10 = sid_KDE
shd_N10 = shd_N
sid_N10 = sid_N

In [38]:
# with orcale MEC:
N = 100

result_dict_oracle = {}

for r in range(N):
    pid = 100+r+1
    save_file = (
            Path("experiments")
            / "results"
            / "MVskew10"
            / f"result_{pid}.pkl"
        )
    with open(save_file, "rb") as file:
            result = pickle.load(file)
    result_dict_oracle[pid] = result


shd_KDE = []
sid_KDE = []
shd_N = []
sid_N = []
for r in range(N):
    pid = r+1
    GT = GT_dict[pid]
    res = result_dict_oracle[pid+100]
    shd_N.append(SHD(target=GT, pred=res["best_graph_N"]))
    sid_N.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_N"]))
    shd_KDE.append(SHD(target=GT, pred=res["best_graph_KDE"]))
    sid_KDE.append(calculate_SID_with_R(W_true=GT, W_est=res["best_graph_KDE"]))

print("SHD KDE: ", np.mean(shd_KDE))
print("SID KDE: ", np.mean(sid_KDE))
print("SHD N: ", np.mean(shd_N))
print("SID N: ", np.mean(sid_N))

SHD KDE:  1.08
SID KDE:  3.31
SHD N:  0.04
SID N:  0.1


In [39]:
# Check accuracy of nodes that were directed by skewd_MV when using the oracle MEC:
from src.graphutils import undirected_edges, correct_edges
undir_edges = {}
cor_edges = {}
accs = []
total_no_edges = 0
correct_no_edges = 0

for r in range(N):
    pid = r+1
    GT_graph = GT_dict[pid]
    oracle_MEC = result_dict_oracle[pid+100]["MEC"]
    predicted_oracle_based = result_dict_oracle[pid+100]["best_graph_N"]
    undir_edges[r] = undirected_edges(oracle_MEC)
    cor_edges[r] = correct_edges(edges=undir_edges[r], GT_graph=GT_graph, pred_graph=predicted_oracle_based)
    if len(undir_edges[r]) != 0:
        accs.append(len(cor_edges[r])/len(undir_edges[r]))
        if(len(cor_edges[r])/len(undir_edges[r])) != 1:
            print(r)
    total_no_edges += len(undir_edges[r])
    correct_no_edges += len(cor_edges[r])


46
78


In [40]:
np.mean(accs)

0.9971504559270518

In [41]:
correct_no_edges/total_no_edges

0.9942528735632183

In [42]:
total_no_edges/100 # 2.57 edges per setting that are directed by skewd w/o PC

3.48

In [43]:
shd_KDE10_O = shd_KDE
sid_KDE10_O = sid_KDE
shd_N10_O = shd_N
sid_N10_O = sid_N

In [44]:
shd_rows = [["Method"] + ["d=6"]+ ["d=10"]]
sid_rows = [["Method"] + ["d=6"] + ["d=10"]]

shd_mv = ["SkewD-MV"]
shd_mv_k = ["SkewD-MV-K"]
shd_mv_O = ["SkewD-MV-O"]
shd_mv_k_O = ["SkewD-MV-K-O"]

sid_mv = ["SkewD-MV"]
sid_mv_k = ["SkewD-MV-K"]
sid_mv_O = ["SkewD-MV-O"]
sid_mv_k_O = ["SkewD-MV-K-O"]

shd_mv.append(np.mean(shd_N6))
shd_mv.append(np.mean(shd_N10))

shd_mv_k.append(np.mean(shd_KDE6))
shd_mv_k.append(np.mean(shd_KDE10))

shd_mv_O.append(np.mean(shd_N6_O))
shd_mv_O.append(np.mean(shd_N10_O))

shd_mv_k_O.append(np.mean(shd_KDE6_O))
shd_mv_k_O.append(np.mean(shd_KDE10_O))

sid_mv.append(np.mean(sid_N6))
sid_mv.append(np.mean(sid_N10))

sid_mv_k.append(np.mean(sid_KDE6))
sid_mv_k.append(np.mean(sid_KDE10))

sid_mv_O.append(np.mean(sid_N6_O))
sid_mv_O.append(np.mean(sid_N10_O))

sid_mv_k_O.append(np.mean(sid_KDE6_O))
sid_mv_k_O.append(np.mean(sid_KDE10_O))

shd_rows += [shd_mv, shd_mv_O, shd_mv_k, shd_mv_k_O]
sid_rows += [sid_mv, sid_mv_O, sid_mv_k, sid_mv_k_O]

with open("experiments/results/skewd_shd.csv", "w", newline="") as fshd, open("experiments/results/skewd_sid.csv", "w", newline="") as fsid:
    writer_shd = csv.writer(fshd)
    writer_sid = csv.writer(fsid)
    writer_shd.writerows(shd_rows)
    writer_sid.writerows(sid_rows)